# 🏗️ Currency-Normalized Sales Intelligence — End-to-End DE Project

**Stack:** `dlt` → `DuckDB` → `dbt` → `Evidence.dev`
**Format:** a sprint-by-sprint build. This is a *brief and a backlog*, not a tutorial.

> ⚠️ **How this notebook works.** Each phase is a real sprint ticket: a goal, tasks, embedded
> production-style problems, and code scaffolding (stubs, configs) — **never the solution**.
> You implement; the notebook only points you at what a senior engineer would worry about.
>
> **Contents:** Phase 0 brief · 1 profiling · 2 dlt ingestion · 3 staging · 4 dbt transforms/tests ·
> 5 dimensional marts (SCD) · 6 Evidence dashboard · 7 portfolio polish.

## 📌 Phase 0 — Project Brief

### The business problem
**NorthWind Global** sells one catalog across several countries and bills customers in **local currency**.
Today the analytics team stitches together a manual Excel export plus ad-hoc currency conversion in
spreadsheets. The result: finance and regional managers **cannot agree on a single revenue number**,
margin is unknowable across regions, and nobody can see how customer segments or product prices shifted
over time. You replace the spreadsheet chaos with a **trustworthy, automated, historically-correct** pipeline.

### Questions the business must answer
- **Total revenue in USD** by month, region, and product category.
- **Gross margin** by category once everything is one currency.
- How a **customer's segment / a product's price changed over time** — and what was true *as of* an order.
- Which orders are **returns, corrections, or duplicates**, and are we double-counting them?

### Grain & rules (lock before building)
- **Fact grain:** one row per *order line* (order × product), de-duplicated, currency-normalized.
- **Conformed currency:** USD at the FX rate **as of the order date** — not today's rate.
- **Late data is expected:** corrected order lines and back-dated FX rates arrive after first load.

### Project "done"
`dbt build` passes tests, produces `fct_sales` + conformed SCD dimensions in DuckDB, and an Evidence
dashboard a regional manager can read without asking you a question.

## 🔌 The three data sources

| # | Source | Gives you | Why |
|---|--------|-----------|-----|
| **A** | `raw_sales_export.xlsx` (+ `_v2`) **dirty** | order lines, ~16 cols | messy operational reality + **schema drift** between v1/v2 |
| **B** | **Frankfurter** `https://api.frankfurter.dev` | daily ECB rates, historical + ranges, **no key** | currency normalization, incremental load, late-arriving rates |
| **C** | **REST Countries** `https://restcountries.com` | region/subregion, ISO codes, currency per country | enrichment + an **unreliable-key join** against messy country text |

**Source A:** two files are attached — `raw_sales_export.xlsx` (v1) and `raw_sales_export_v2.xlsx` (a later
feed). Put both in `data/raw/`. They are *intentionally* messy; do not look for a data dictionary.
The v2 file's columns **do not match v1** — that mismatch is your schema-drift exercise (Phase 2).

**Source C caveat (a real vendor problem):** REST Countries is mid-migration. Keyless `/v3.1` still answers
but a `/v5` requiring a free key is rolling out. Decide: pin v3.1, register a v5 key, or **snapshot a static
country reference** so a third-party deprecation can't break your pipeline. That decision *is* engineering.

## 🗺️ Target architecture

```
            ┌─────────────┐      ┌──────────────┐      ┌───────────────────────┐
  Excel  ──▶│  dlt        │ ───▶ │  DuckDB      │ ───▶ │  dbt                  │ ──▶ Evidence.dev
  FX API ──▶│  ingestion  │      │  raw schema  │      │  staging→intermediate │     dashboard
  Country─▶ │ (incremental│      │ (load tables │      │  →marts (facts/dims,  │
            │  + schema   │      │  + lineage)  │      │   SCD2, tests, docs)  │
            │  evolution) │      └──────────────┘      └───────────────────────┘
            └─────────────┘
  land untouched ─────────▶ cast/rename/clean ─────▶ dedup, conform FX, SCD, model facts/dims
```

**Layering contract:** `raw` → `staging` (1:1 clean/cast) → `intermediate` (joins, dedup, conform) →
`marts` (facts + SCD dims). A mart never reads `raw` directly.

### Repo layout
```
northwind-de/
├── data/raw/{raw_sales_export.xlsx, raw_sales_export_v2.xlsx}
├── ingestion/{sales_excel_pipeline.py, fx_api_pipeline.py, countries_pipeline.py}
├── warehouse/northwind.duckdb
├── dbt_northwind/{models/{staging,intermediate,marts}, snapshots, tests, dbt_project.yml}
├── reports/                      # Evidence app
├── PROFILING.md                  # Phase 1 deliverable
└── requirements.txt
```

---
# 🧭 Phase 1 — Source exploration & profiling

> **Sprint goal:** understand all three sources well enough to write a *data contract*. You write **zero**
> cleaning logic this sprint — observe, measure, document. Cleaning is Phase 3.

**Definition of done:**
- [ ] `PROFILING.md` listing every data-quality issue with evidence (counts, examples).
- [ ] A **draft data contract** per source: column → intended type, nullability, key, known issues.
- [ ] A documented decision on REST Countries v3.1-vs-v5.
- [ ] A stated **dedup key** and **incremental key** for each source.

In [ ]:
from pathlib import Path
# TODO: resolve relative to repo root, add real existence checks + friendly errors.
RAW_XLSX   = Path("data/raw/raw_sales_export.xlsx")
RAW_XLSX_2 = Path("data/raw/raw_sales_export_v2.xlsx")
DUCKDB     = Path("warehouse/northwind.duckdb")
FX_BASE    = "https://api.frankfurter.dev/v1"   # latest | {date} | {start}..{end}
COUNTRIES  = "https://restcountries.com"        # /v3.1 (keyless, deprecating) vs /v5 (key)

In [ ]:
import pandas as pd

def load_raw_excel(path) -> pd.DataFrame:
    """Load the export with NOTHING coerced or cleaned.
    Think: which sheet holds orders? does the table start on row 1, or is junk above it?
    Which single read option stops pandas inferring dtypes and masking the mess?"""
    raise NotImplementedError

# raw = load_raw_excel(RAW_XLSX); raw.head(20)

In [ ]:
def column_overview(df) -> pd.DataFrame:
    """Per column: non-null count, null %, n_unique, stored dtype, 3 sample values."""
    raise NotImplementedError

def find_mixed_types(df, col) -> dict:
    """Bucket a column's values by actual Python type / parse-ability."""
    raise NotImplementedError

def date_parse_report(df, col) -> dict:
    """How many values parse under ONE strategy? Show the failures (there is >1 format, and a non-date)."""
    raise NotImplementedError

def duplicate_report(df, key_cols) -> pd.DataFrame:
    """Full-row dups AND key-only dups separately — one is noise, one is a correction/late row."""
    raise NotImplementedError

def currency_text_report(df, *cols) -> dict:
    """Catalogue symbols/separators/decimal-conventions hiding in money/qty text. Inventory only."""
    raise NotImplementedError

### Task 1.4 — The hunt (challenges — described, **not** solved)
Prove each with a count or example; **fix nothing yet**.
1. **The header isn't on row 1.** Find the real header row and what sits above it.
2. **Order IDs aren't clean keys.** Unique? Consistently formatted? What must you normalize for a PK to hold?
3. **Dates wear disguises.** Multiple formats + at least one non-date. Which format is *ambiguous* (day/month swap)?
4. **Money is text.** Symbols + separators, and the **decimal convention differs by row**. Why is `"2.300,00"` a trap?
5. **Mojibake.** Some names are corrupted. Which encoding round-trip caused it? How do you *detect* it programmatically?
6. **Country is an unreliable key.** Count distinct spellings of the same place; start a crosswalk — you'll join Source C on this.
7. **Currency may be blank.** Recoverable from what? State the rule (implement later).
8. **Returns vs junk.** Negative *and* non-numeric quantities appear. Which are real business events?
9. **Duplicates with intent.** An exact dup and a *near*-dup (same order, changed amount). Which is "late-arriving corrected data"?
10. **Trailing junk.** Non-tabular content sits at the bottom / in a merged cell. How will your loader avoid ingesting it?

In [ ]:
import requests
def peek_fx(url: str) -> dict:
    """Fetch+parse FX JSON. Exploration only.
    Probe: /latest?base=USD ; /2024-01-15?base=USD&symbols=EUR,GBP ; /2024-01-01..2024-03-31?base=USD&symbols=EUR
    Answer in PROFILING.md:
      - natural incremental cursor?
      - 2024-01-01 returns a rate dated ...? (non-trading day) → impact on joining FX to a weekend order?
      - 'today' is revisable → why is blind full-refresh risky, and what about late-arriving rates?
      - convert at ingest or in dbt? justify."""
    raise NotImplementedError

In [ ]:
def peek_countries(url: str) -> list:
    """Fetch country reference (v3.1/all?fields=name,cca2,cca3,region,subregion,currencies — or v5+key).
    Decide & record:
      - match key: name vs ISO alpha-2/3? build crosswalk messy-spelling -> ISO.
      - how many of YOUR distinct spellings fail an exact match? list them.
      - unmatched rows: fail / quarantine / hand-map? pick one, justify.
      - vendor risk: pin v3.1 / take v5 key / snapshot static CSV? decide."""
    raise NotImplementedError

### 🧠 Phase 1 reflection (in `PROFILING.md`)
- Primary key for the sales fact, and what must hold for it to be valid?
- Dedup rule in one sentence — which duplicate type does it intentionally *keep*?
- Incremental cursor for each source?
- Which issues fix in **staging** (mechanical) vs **intermediate** (cross-source/business)?
- Name the first three **dbt tests** you already know you'll need.

---
# 📥 Phase 2 — Raw ingestion with dlt (Excel + APIs)

> **Sprint goal:** land all three sources into DuckDB `raw` with **dlt**, using **incremental loading** and
> **schema evolution**. Still **no cleaning** — the only transformation allowed is dropping non-data junk rows
> so the table is rectangular. Conversion, parsing, conforming all happen later.

**Definition of done:**
- [ ] `raw.sales`, `raw.fx_rates`, `raw.countries` exist in `warehouse/northwind.duckdb`.
- [ ] FX loads **incrementally** by date (re-running doesn't re-pull all history).
- [ ] Loading `_v2` next to `_v1` is handled deliberately (you can explain what dlt did to the schema).
- [ ] `_dlt_load_id` lineage is queryable; you can answer "which load did this row come from?".

### Task 2.1 — Pipeline skeleton
One pipeline, DuckDB destination, dataset `raw`. Keep secrets/config out of code.

In [ ]:
import dlt

# ingestion/_pipeline.py
def build_pipeline():
    """Return a dlt pipeline writing to the DuckDB file at DUCKDB, dataset_name='raw'.
    Decide: one pipeline with three resources, or three pipelines? Justify in a comment.
    (dlt creates _dlt_loads / _dlt_load_id automatically — you'll lean on those for lineage.)"""
    raise NotImplementedError

### Task 2.2 — Excel resource (Source A)
A dlt resource that yields order-line dicts from the Excel.
**Challenges:**
- Your generator must **skip the junk title rows and the trailing merged note** — yield only real rows.
- Pick `write_disposition`: `replace` vs `append` vs `merge`. For a periodic *full* export, which is correct, and what `primary_key` makes `merge` idempotent given the messy Order IDs?
- You will load **two files with different schemas** (Task 2.5). Design the resource so that's possible.

In [ ]:
@dlt.resource(name="sales", write_disposition="???", primary_key="???")
def sales_resource(path):
    """Yield one dict per order line from the Excel, raw values untouched (no casting).
    Only structural fixing allowed: start at the true header row, stop before trailing junk.
    Do NOT parse dates/money here — that's staging."""
    raise NotImplementedError

### Task 2.3 — FX resource with **incremental loading** (Source B)
Frankfurter returns `{date: {symbol: rate}}`. Make dlt pull only *new* dates after the first run.
**Challenges:**
- Use `dlt.sources.incremental` on a **date cursor**. What's the initial backfill window, and where does the cursor start?
- ECB skips weekends/holidays → a requested date can return a rate dated earlier. Do you store the **requested** date, the **returned** date, or both? (Your Phase 4 as-of join depends on this.)
- "Today" is revisable. How do you let a recent window be **re-fetched** without duplicating rows? (Hint: `merge` + a composite key, and a small lookback — but you decide the key.)

In [ ]:
@dlt.resource(name="fx_rates", write_disposition="merge", primary_key=("???",))
def fx_resource(
    base: str = "USD",
    symbols = ("EUR", "GBP", "JPY", "CAD"),
    cursor = dlt.sources.incremental("rate_date", initial_value="2024-01-01"),
):
    """Fetch Frankfurter ranges and yield one row per (date, base, symbol, rate).
    Use `cursor.last_value` to request only dates you don't have yet, with a small lookback
    so revisable recent rates get corrected. Flatten the nested JSON into tidy rows."""
    raise NotImplementedError

### Task 2.4 — Countries resource (Source C)
Reference data — small and slow-changing.
**Challenges:** full `replace` each run, or `merge` on `cca3`? If you chose the *static snapshot* strategy in
Phase 1, this resource reads your committed CSV instead of the API — wire that in and note the trade-off.

In [ ]:
@dlt.resource(name="countries", write_disposition="???", primary_key="cca3")
def countries_resource():
    """Yield one row per country with name(s), cca2, cca3, region, subregion, currency code(s).
    Flatten the nested currencies/name objects into columns you can actually join on."""
    raise NotImplementedError

### Task 2.5 — 🧨 Schema-drift drill (required)
Load `raw_sales_export.xlsx`, inspect the schema, then load `raw_sales_export_v2.xlsx`.
**Observe and explain (write it down):**
- `Amount` became `Total Amount`; `Segment` disappeared; `Order Channel` and `Tax` appeared.
- What did dlt do? (New columns added to the table? The rename treated as a *new* column with the old one going NULL?)
- This is the trap: **dlt won't know two differently-named columns mean the same thing.** Where do you reconcile that — at ingest (normalize headers before yielding) or in staging (map both to one field)? Pick one, justify, and make re-running deterministic.
- Inspect `dlt`'s stored schema version. How would you alert on *unexpected* drift in production?

In [ ]:
# After both loads, interrogate what happened:
import duckdb
# con = duckdb.connect(str(DUCKDB))
# con.sql("DESCRIBE raw.sales").show()
# con.sql("SELECT _dlt_load_id, count(*) FROM raw.sales GROUP BY 1").show()
# Questions: which columns are NULL for v1 rows? for v2 rows? what does that imply for staging?

### 🧠 Phase 2 reflection
- Exact `write_disposition` + key for each of the three resources, and why each is *idempotent on re-run*.
- Your incremental cursor for FX, and how a late-arriving/revised rate gets corrected without dupes.
- Your schema-drift reconciliation point, in one sentence.

---
# 🧱 Phase 3 — Staging layer (light cleaning, casting, renaming)

> **Sprint goal:** one **1:1 staging model per raw table** that is mechanically clean: correct types,
> snake_case names, trimmed strings, parsed dates/money, normalized enums. **No joins, no business logic,
> no dedup** — staging is boring on purpose. Materialize as **views**.

This is the first dbt-managed layer. Set the project up now.

**Definition of done:**
- [ ] `dbt_northwind` runs against DuckDB; `dbt debug` is green.
- [ ] `stg_sales`, `stg_fx_rates`, `stg_countries` build as views.
- [ ] Every column is correctly typed; the v1/v2 column mismatch is reconciled to **one** schema here.

### Task 3.1 — dbt + DuckDB project config (scaffold)
```yaml
# dbt_northwind/profiles.yml
northwind:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: ../warehouse/northwind.duckdb   # TODO: confirm relative path from dbt project dir
      schema: main
      threads: 4
```
```yaml
# dbt_northwind/dbt_project.yml
name: northwind
profile: northwind
models:
  northwind:
    staging:      {+materialized: view,  +schema: staging}
    intermediate: {+materialized: view,  +schema: intermediate}
    marts:        {+materialized: table, +schema: marts}
```
> ⚠️ DuckDB is a single-file embedded DB. dbt and Evidence both want to open it. If something holds a write
> lock, the other gets `IO Error: Could not set lock`. Note now how you'll avoid concurrent writers (Phase 6).

### Task 3.2 — `stg_sales` (the hard one)
File: `models/staging/stg_sales.sql`. Reference raw via a dbt **source**.
**Challenges (describe → implement; the notebook gives none of the SQL):**
- **Dates:** parse a column holding *several* formats into one `DATE`. The ambiguous `dd/mm` vs `mm/dd` rows
  can silently swap — decide a rule and make wrong parses **fail loudly**, not silently.
- **Money:** turn `"$1,234.50"` and `"2.300,00"` into a numeric. The decimal convention differs by row — a blind
  "strip non-digits" corrupts the European value. How do you disambiguate?
- **Mojibake:** can you repair `"CafÃ© Ã–lsson"` in SQL, or is this better fixed at ingest? Decide and be consistent.
- **Enums:** collapse `Consumer/consumer/CONSUMER` and country spellings to canonical values (full ISO crosswalk is Phase 4 — here just trim/upper/standardize obvious cases).
- **v1/v2 reconciliation:** map `Total Amount`→`amount`, recover/flag dropped `Segment`, surface new `Order Channel`/`Tax`.
- **Keep, don't drop, bad rows:** add an `is_quarantined` flag + reason rather than deleting — downstream tests need to see them.

In [ ]:
# models/staging/stg_sales.sql  (skeleton — fill the CTE bodies)
"""
with source as (select * from {{ source('raw', 'sales') }}),
renamed as (
    -- TODO: snake_case + reconcile v1/v2 column names to ONE schema
    select * from source
),
typed as (
    -- TODO: parse order_date/ship_date -> date ; quantity -> int ; amount/unit_price -> numeric
    -- TODO: standardize currency code ; trim names ; flag is_quarantined + quarantine_reason
    select * from renamed
)
select * from typed
"""

### Task 3.3 — `stg_fx_rates` & `stg_countries`
- `stg_fx_rates`: enforce `(rate_date date, base, symbol, rate numeric)`; ensure the **requested vs returned**
  date distinction from Phase 2 survives.
- `stg_countries`: one row per country, ISO codes + region + currency, ready to be a join target.

### 🧠 Phase 3 reflection
- Your date-parse failure policy: what happens to an unparseable date — null, error, or quarantine?
- Your money-disambiguation rule, stated precisely.
- Why staging is views, not tables (cost vs. freshness vs. lineage).

---
# 🔁 Phase 4 — Transformation layer (intermediate, tests, incremental)

> **Sprint goal:** the *engineering-heavy* sprint. Build intermediate models that **dedup**, resolve
> **late-arriving corrections**, and **conform currency** by joining FX as-of the order date. Make at least one
> model **incremental**. Wrap everything in **tests + docs**.

**Definition of done:**
- [ ] `int_sales_deduped`, `int_sales_conformed` build.
- [ ] One incremental model with correct `unique_key` + late-arriving handling.
- [ ] `sources.yml` with freshness; generic + **custom** tests; `dbt build` is green; `dbt docs generate` works.

### Task 4.1 — Sources, freshness, source tests (scaffold)
```yaml
# models/staging/_sources.yml
version: 2
sources:
  - name: raw
    schema: raw
    tables:
      - name: sales
        loaded_at_field: _dlt_load_id   # TODO: a real timestamp; decide freshness policy
        freshness: {warn_after: {count: 36, period: hour}, error_after: {count: 72, period: hour}}
      - name: fx_rates
      - name: countries
```

### Task 4.2 — `int_sales_deduped` — **deduplication + late-arriving data**
**Challenges:**
- Exact duplicates: collapse them. Easy.
- *Near*-duplicates (same Order ID, corrected `amount`): keep the **latest correction**, not the original. What
  ordering key proves which row is newest? (You may have to derive one — there's no clean `updated_at`.)
- This is a textbook **late-arriving fact** problem: the correction can land in a *later* load than the original.
  Your dedup must survive across loads (use `_dlt_load_id` / a load timestamp).
- Returns (negative quantity) are **not** duplicates — make sure your rule doesn't eat them.

In [ ]:
# models/intermediate/int_sales_deduped.sql (skeleton)
"""
with cleaned as (select * from {{ ref('stg_sales') }} where not is_quarantined),
ranked as (
    -- TODO: row_number() over (partition by <natural key> order by <recency key> desc)
    -- decide the natural key AND the recency key; justify both in the model description
    select *, 1 as rn from cleaned
)
select * from ranked where rn = 1
"""

### Task 4.3 — `int_sales_conformed` — **as-of FX join + unreliable country key**
**Challenges:**
- Convert each line to USD using the rate **effective on the order date**. Frankfurter has gaps (weekends).
  Implement an **as-of join**: take the most recent rate ≤ order_date. A naive equi-join on date drops weekend orders.
- Some orders have a currency you have **no rate** for, or a date before your FX history. Route them to a
  **quarantine**/exception table — don't silently null the revenue.
- Join messy `country` to `stg_countries` via your Phase-1 crosswalk. Count and surface unmatched countries.

In [ ]:
# models/intermediate/int_sales_conformed.sql (skeleton)
"""
with s as (select * from {{ ref('int_sales_deduped') }}),
fx as (select * from {{ ref('stg_fx_rates') }} where base = 'USD'),
asof as (
    -- TODO: for each sale, pick the latest fx row with rate_date <= order_date and matching symbol=currency
    -- (DuckDB supports ASOF JOIN — or do it with a window. Your call.)
    select * from s
)
-- TODO: amount_usd = amount / rate (or * rate — get the direction right and TEST it)
-- TODO: left-join countries on the crosswalk; flag unmatched
select * from asof
"""

### Task 4.4 — Make it **incremental**
Pick the model that grows (conformed sales) and convert it.
**Challenges:**
- `materialized='incremental'`, a real `unique_key`, and an `is_incremental()` filter on a load timestamp.
- **Late data breaks naive incrementals:** if you only process rows newer than `max(loaded_at)`, a back-dated
  correction is missed. Add a **lookback window** (reprocess the last N days/loads) and an `incremental_strategy`
  (`merge`/`delete+insert`) that overwrites corrected rows. State your N and why.

In [ ]:
# top of int_sales_conformed.sql, if incremental:
"""
{{ config(materialized='incremental', unique_key='order_line_id',
          incremental_strategy='delete+insert') }}
-- ...
{% if is_incremental() %}
  -- TODO: where loaded_at >= (select max(loaded_at) from {{ this }}) - interval '<N> days'
{% endif %}
"""

### Task 4.5 — Tests: generic + **custom** (data quality)
In `_models.yml` add generic tests; in `tests/` add singular SQL tests.
**At minimum:**
- Generic: `unique` + `not_null` on the order-line key; `relationships` from sales→customers/products;
  `accepted_values` on segment/channel/currency.
- Custom singular tests (each is a SELECT that must return **zero rows**):
  - no order with `order_date` in the future;
  - no conformed row where `currency` had a rate but `amount_usd` is null;
  - revenue reconciliation: `sum(amount_usd)` within tolerance of an independent recompute;
  - every distinct source `country` either maps to ISO or appears in the quarantine table.
- Consider `dbt_utils` (`unique_combination_of_columns`, `expression_is_true`).

In [ ]:
# tests/assert_no_future_orders.sql
"select * from {{ ref('int_sales_conformed') }} where order_date > current_date"
# tests/assert_usd_present_when_rate_exists.sql  -> you write it
# tests/assert_country_mapped_or_quarantined.sql -> you write it

### Task 4.6 — Docs
Add `description:` to models/columns; run `dbt docs generate && dbt docs serve`. The lineage graph is a
portfolio screenshot later.

### 🧠 Phase 4 reflection
- Your dedup natural key + recency key, and how late corrections win.
- Your as-of FX rule in one sentence; what happens to un-rateable rows.
- Your incremental lookback N and strategy, and the failure it prevents.
- The three custom tests you wrote and the bug each would catch.

---
# ⭐ Phase 5 — Mart layer (dimensional modeling + SCD)

> **Sprint goal:** a clean **star schema**: a fact at order-line grain surrounded by conformed dimensions,
> including **slowly changing dimensions** so the business can ask "what was true *as of* this order?".

**Definition of done:**
- [ ] `dim_date`, `dim_customer` (SCD2), `dim_product` (SCD2 for price), `dim_geography`, `dim_currency`.
- [ ] `fct_sales` at order-line grain, all measures in USD, FKs resolved **point-in-time** to the SCD dims.
- [ ] `dbt build` green with tests on the marts.

### Task 5.1 — `dim_date`
Generate a contiguous calendar covering your order/ship/FX span; attributes (year, month, quarter, is_weekend).
**Challenge:** which date(s) does the fact join on — order, ship, both as role-playing dimensions?

### Task 5.2 — `dim_customer` as **SCD Type 2**
Customers change segment / country over time and you must preserve history.
**Challenges (the crux of this sprint):**
- There is **no reliable change timestamp** in the source. How do you detect a changed attribute and stamp a
  `valid_from`? (dbt **snapshots** with `check` strategy are the idiomatic tool — scaffold below.)
- Generate a **surrogate key** per version; add `valid_from`, `valid_to`, `is_current`.
- **Late-arriving dimension:** a sale can reference a customer version you haven't recorded yet. Decide the policy
  (insert an inferred version? route to an unknown-member key `-1`?).

In [ ]:
# snapshots/customer_snapshot.sql (scaffold)
"""
{% snapshot customer_snapshot %}
{{ config(target_schema='snapshots', unique_key='customer_id',
          strategy='check', check_cols=['segment','country','customer_name']) }}
select customer_id, customer_name, segment, country
from {{ ref('stg_sales') }}      -- TODO: dedupe to one row per customer per load first
{% endsnapshot %}
"""
# dim_customer then reads the snapshot and shapes valid_from/valid_to/is_current + surrogate_key

### Task 5.3 — `dim_product`
Price changes over time → **SCD2 on price** (or argue SCD1 and lose history — your call, justify it).
Same surrogate-key + validity-window pattern as customers.

### Task 5.4 — `fct_sales` (the point-in-time join)
**Challenges:**
- Grain: one row per deduped order line. Carry a stable `order_line_id`.
- Resolve FKs **as of order_date** against the SCD2 dims: join where `order_date between valid_from and valid_to`.
  An equi-join on the natural key alone gives you *today's* version — wrong for history. This is the whole point of SCD2.
- Measures in USD: `quantity`, `gross_amount_usd`, `discount_usd`, `net_amount_usd`. If you have no cost source,
  say so and leave margin as a documented gap (don't fabricate it).
- Returns (negative qty) flow through as negative measures — verify they net correctly.
- Add an **unknown-member** row (`-1`) to each dim so the fact never drops a row on a failed lookup.

In [ ]:
# models/marts/fct_sales.sql (skeleton)
"""
with sales as (select * from {{ ref('int_sales_conformed') }}),
cust as (select * from {{ ref('dim_customer') }}),
prod as (select * from {{ ref('dim_product') }})
-- TODO: point-in-time join sales -> cust on customer_id AND order_date in [valid_from, valid_to]
-- TODO: same for product ; coalesce to -1 unknown member on miss
-- TODO: select surrogate FKs + USD measures at order-line grain
select 1
"""

### 🧠 Phase 5 reflection
- How you detect attribute changes without a source timestamp, and where `valid_from` comes from.
- Your point-in-time join predicate, written out.
- Your unknown-member strategy and which real failure it hides vs. surfaces.

---
# 📊 Phase 6 — Evidence.dev dashboard

> **Sprint goal:** a small, honest BI app reading **straight from DuckDB marts**, that a regional manager
> can use unaided — including a page that proves your data quality.

**Definition of done:**
- [ ] Evidence connects to `northwind.duckdb` and renders without errors.
- [ ] Revenue (USD) by month/region/category; margin-or-gap; a **segment-over-time** view that *uses* SCD2.
- [ ] A **data-quality page**: quarantined-row counts, unmatched countries, test status.
- [ ] A working dropdown/parameter.

### Task 6.1 — Connect (scaffold)
```bash
npx degit evidence-dev/template reports && cd reports && npm install
```
```yaml
# reports/sources/northwind/connection.yaml
name: northwind
type: duckdb
options: { filename: ../../warehouse/northwind.duckdb }   # TODO: confirm relative path
```
> ⚠️ **The lock problem returns.** If `dbt build` is running (write lock) Evidence's read can fail, and vice
> versa. Options: build first then serve; point Evidence at a **read-only copy** of the file; or export marts to
> parquet for the dashboard. Pick one and write down the trade-off.

### Task 6.2 — Pages (each is a `.md` with SQL + a chart component)
```markdown
<!-- reports/pages/index.md -->
# NorthWind Global — Sales (USD)
'''sql revenue_by_month
select date_trunc('month', order_date) as month,
       sum(net_amount_usd) as revenue_usd
from northwind.marts.fct_sales        -- TODO: confirm schema/table name
group by 1 order by 1
'''
<LineChart data={revenue_by_month} x=month y=revenue_usd />
```
**Build these pages:**
- **Overview:** revenue USD by month, region, category (use `dim_geography`, `dim_date`).
- **Segment over time:** revenue by customer segment by month — and prove it reads the **SCD2** segment
  (a customer who switched segments should appear under *both*, in the right periods).
- **Data quality:** counts of quarantined sales, unmatched countries, latest `dbt test` result. This page is what
  makes the project credible to a reviewer.
- A **parameter/dropdown** (e.g., region filter) on the overview.

### 🧠 Phase 6 reflection
- Your DuckDB concurrency choice (build-then-serve / read-only copy / parquet) and why.
- One number on the dashboard you could defend end-to-end, naming every transform it passed through.

---
# ✨ Phase 7 — Portfolio polish (what makes it *GitHub-worthy*)

Not a sprint of features — a sprint of **credibility**.
- [ ] `README.md`: the business problem, an architecture diagram, a screenshot of the **dbt lineage graph** and
      the **dashboard**, and a one-command run (`make build` / `dlt run … && dbt build && evidence dev`).
- [ ] A short **"Known issues & decisions"** section — every quarantine rule, the v3.1-vs-v5 call, the FX-direction
      and lookback decisions. Showing judgment beats pretending the data was clean.
- [ ] `dbt docs` published (or screenshot); test count visible.
- [ ] A `data/` note explaining the sources are intentionally messy and how to regenerate FX/country pulls.
- [ ] `.gitignore` the `.duckdb` file and any keys; commit a tiny **sample** instead.

### Stretch (only if the core is green)
- Containerize (`docker-compose`: pipeline + dbt + evidence).
- Orchestrate the daily run (a simple scheduler / GitHub Action) to make incremental loading *real*.
- A `dbt source freshness` gate that fails CI when the feed is stale.

> 🎓 **When all seven phases are green, you have an end-to-end pipeline that ingests messy multi-source data
> incrementally, handles drift/dedup/late-arrival/SCD, is tested and documented, and ends in a dashboard you can
> defend line by line. That's the project — go build it.**